# VLST — TabPFN (tabular foundation model)

## Overview

**TabPFN** ([PriorLabs TabPFN](https://github.com/PriorLabs/TabPFN)) is a transformer-style model trained on synthetic tabular tasks; at **inference** time it performs **in-context learning** on your training rows (no classical hyperparameter grid search). It is often strong on **small/medium** tabular problems, including **imbalanced** settings when using built-in options such as **`balance_probabilities`**.

This notebook is aligned with **`baseline_without_tssi.ipynb`**:

- **Data**: `data/processed/` under the VLST repo (`X_train.npy`, …) from **`preprocessing.ipynb`**. Paths are **auto-detected** from the kernel’s current working directory (works when cwd is `Downloads`, `vlst_repo/code`, etc.). Override with **`MANUAL_VLST_ROOT`** in §2 if needed.
- **Leakage control**: drops **`Time since stent implantation`** (same as other modeling notebooks)
- **Optional SMOTE**: same toggle as baselines (default **off**; TabPFN already targets imbalance via probabilities)
- **Outputs**: metrics and plots under **`../data/result/modeling_tabpfn/`**

### Important notes

1. **Install**: Run §1 (`%pip install ...`) in a **virtualenv kernel** (not system Python — PEP 668). That cell installs **`tabpfn`** and **`imbalanced-learn`** (needed only if **`USE_SMOTE = True`**).
2. **Preprocessing**: TabPFN’s docs often recommend feeding **raw** tabular data and letting the model preprocess. Here we default to the **same scaled / one-hot matrices** as the rest of your pipeline so results are **comparable** to XGBoost/LightGBM notebooks. Set **`USE_RAW_FEATURES = True`** below to build features from **`../data/raw/VLST.csv`** with a **stratified split** (same `random_state` as `preprocessing.ipynb`) — scores will **not** match other notebooks exactly, but inputs are closer to TabPFN’s intended use.
3. **Size / CPU**: Training sets larger than typical pretraining ranges may require **`ignore_pretraining_limits=True`** (set below). On CPU, very large train sets can be impractical.
4. **Cross-validation**: Fitting TabPFN per fold is **expensive**. Optional stratified CV is **off** by default; enable **`RUN_TABPFN_CV`** when you can afford the runtime.

## 1. Install TabPFN (run once per environment)

In [3]:
%pip install -q 'tabpfn>=7.0' 'imbalanced-learn>=0.12'

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try 'pacman -S
    python-xyz', where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Arch-packaged Python package,
    create a virtual environment using 'python -m venv path/to/venv'.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip.
    
    If you wish to install a non-Arch packaged Python application,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. Make sure you have python-pipx
    installed via pacman.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys, subprocess
print(sys.executable)
r = subprocess.run(
    [sys.executable, "-m", "pip", "-V"],
    capture_output=True,
    text=True,
)
print("returncode", r.returncode)
print(r.stdout or r.stderr)

/home/fadia/Downloads/.venv/bin/python
/home/fadia/Downloads/.venv
/usr


cursor: bad option: -m


CalledProcessError: Command '['/home/fadia/Downloads/.venv/bin/python', '-m', 'pip', '-V']' returned non-zero exit status 9.

## 2. Configuration

In [6]:
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

warnings.filterwarnings("ignore")
np.random.seed(42)

# If auto-discovery fails, set this to your clone root (folder that contains data/raw/ and data/processed/).
MANUAL_VLST_ROOT: Path | None = None  # e.g. Path("/home/you/Downloads/vlst_repo")


def _resolve_processed_dir() -> Path:
    if MANUAL_VLST_ROOT is not None:
        p = Path(MANUAL_VLST_ROOT).expanduser().resolve() / "data" / "processed"
        if (p / "X_train.npy").is_file():
            return p
        raise FileNotFoundError(
            f"MANUAL_VLST_ROOT set but missing {p / 'X_train.npy'} — run preprocessing.ipynb first."
        )
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents][:14]:
        for parts in (("data", "processed"), ("vlst_repo", "data", "processed")):
            cand = root.joinpath(*parts)
            if (cand / "X_train.npy").is_file():
                return cand
    raise FileNotFoundError(
        "Could not find data/processed/X_train.npy. Typical fixes: (1) Run code/preprocessing.ipynb "
        "to create vlst_repo/data/processed/. (2) Set MANUAL_VLST_ROOT to your vlst_repo path. "
        f"(3) Set USE_RAW_FEATURES=True if ../data/raw/VLST.csv exists. Current cwd: {cwd}"
    )


PROCESSED_DIR = _resolve_processed_dir()
DATA_ROOT = PROCESSED_DIR.parent.parent  # .../vlst_repo
RAW_PATH = DATA_ROOT / "data" / "raw" / "VLST.csv"
RESULT_DIR = DATA_ROOT / "data" / "result" / "modeling_tabpfn"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using PROCESSED_DIR={PROCESSED_DIR}")
print(f"Using RESULT_DIR={RESULT_DIR}")

TARGET_COL = "Stent thrombosis"
DROP_FEATURES = ["Time since stent implantation"]
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Match preprocessing.ipynb split when rebuilding from raw
USE_RAW_FEATURES = False  # True: TabPFN-style raw table + same random_state split

USE_SMOTE = False  # True requires imbalanced-learn (§1)
IGNORE_PRETRAINING_LIMITS = True  # needed for n_train outside default pretraining range / CPU limits
BALANCE_PROBABILITIES = True  # helps under strong class imbalance (VLST)
DEVICE = "auto"  # "cuda", "cpu", or "auto"
N_ESTIMATORS = 8  # TabPFN ensemble of forward passes (not traditional trees)

RUN_TABPFN_CV = False  # True: stratified K-fold (slow)
CV_SPLITS = 3

Using PROCESSED_DIR=/home/fadia/Downloads/vlst_repo/data/processed
Using RESULT_DIR=/home/fadia/Downloads/vlst_repo/data/result/modeling_tabpfn


## 3. Load data (processed or raw)

In [4]:
def drop_tssi_from_arrays(X_tr, X_te, feature_names_list):
    fn = np.array(feature_names_list)
    drop_mask = np.isin(fn, DROP_FEATURES)
    if not drop_mask.any():
        raise ValueError(
            f"None of {DROP_FEATURES} found in feature_names; check feature_names.csv."
        )
    keep = ~drop_mask
    return X_tr[:, keep], X_te[:, keep], fn[keep].tolist()


if not USE_RAW_FEATURES:
    X_train = np.load(PROCESSED_DIR / "X_train.npy")
    X_test = np.load(PROCESSED_DIR / "X_test.npy")
    y_train = np.load(PROCESSED_DIR / "y_train.npy")
    y_test = np.load(PROCESSED_DIR / "y_test.npy")
    feature_names = pd.read_csv(PROCESSED_DIR / "feature_names.csv")[
        "feature_name"
    ].tolist()
    X_train, X_test, feature_names = drop_tssi_from_arrays(
        X_train, X_test, feature_names
    )
    print("Using preprocessed .npy arrays (comparable to other modeling notebooks).")
else:
    if not RAW_PATH.is_file():
        raise FileNotFoundError(
            f"Raw CSV not found: {RAW_PATH}. Place VLST.csv there or set USE_RAW_FEATURES=False "
            "and run preprocessing.ipynb."
        )
    df = pd.read_csv(RAW_PATH)
    for col in ("NO.", "Name"):
        if col in df.columns:
            df = df.drop(columns=[col])
    y = df[TARGET_COL].values
    X_df = df.drop(columns=[TARGET_COL])
    X_df = X_df.apply(pd.to_numeric, errors="ignore")
    cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
    for c in cat_cols:
        X_df[c] = pd.Categorical(X_df[c].astype("string").fillna("__nan__")).codes
    for c in [x for x in DROP_FEATURES if x in X_df.columns]:
        X_df = X_df.drop(columns=[c])
    X_raw = np.nan_to_num(X_df.values.astype(np.float64), nan=0.0)
    feature_names = list(X_df.columns)
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    print("Using raw-derived numeric matrix + stratified split (TabPFN-oriented).")

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {len(feature_names)}")
print(
    f"Train target: 0={np.sum(y_train == 0)}, 1={np.sum(y_train == 1)} | "
    f"Test: 0={np.sum(y_test == 0)}, 1={np.sum(y_test == 1)}"
)

Using preprocessed .npy arrays (comparable to other modeling notebooks).
Train: (4148, 173), Test: (1037, 173)
Features: 173
Train target: 0=4074, 1=74 | Test: 0=1019, 1=18


## 4. Optional SMOTE (training set only)

In [7]:
import importlib

X_train_before_smote = X_train.copy()
y_train_before_smote = y_train.copy()

if USE_SMOTE:
    try:
        _smote_mod = importlib.import_module("imblearn.over_sampling")
        SMOTE = getattr(_smote_mod, "SMOTE")
    except (ImportError, AttributeError) as e:
        raise ImportError(
            "USE_SMOTE=True requires imbalanced-learn. Run §1 in a venv: "
            '%pip install imbalanced-learn'
        ) from e

    n_minority = (y_train == 1).sum()
    k = min(5, n_minority - 1) if n_minority > 1 else 1
    if k < 1:
        print("SMOTE skipped: minority class too small.")
    else:
        smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        print(
            f"SMOTE applied: train -> {X_train.shape[0]} rows "
            f"(0={(y_train == 0).sum()}, 1={(y_train == 1).sum()})"
        )
else:
    print("SMOTE off (recommended default for TabPFN).")

ModuleNotFoundError: No module named 'imblearn'

## 5. Fit TabPFN and evaluate (hold-out)

In [ ]:
from tabpfn import TabPFNClassifier

clf = TabPFNClassifier(
    n_estimators=N_ESTIMATORS,
    device=DEVICE,
    ignore_pretraining_limits=IGNORE_PRETRAINING_LIMITS,
    balance_probabilities=BALANCE_PROBABILITIES,
    random_state=RANDOM_STATE,
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

metrics = {
    "accuracy": (y_pred == y_test).mean(),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob),
    "pr_auc": average_precision_score(y_test, y_prob),
}
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

pd.Series(metrics).to_csv(RESULT_DIR / "tabpfn_holdout_metrics.csv")
joblib.dump(
    {"feature_names": feature_names, "use_raw_features": USE_RAW_FEATURES},
    RESULT_DIR / "tabpfn_run_meta.joblib",
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name="TabPFN")
axes[0].set_title("ROC")
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name="TabPFN")
axes[1].set_title("Precision–Recall")
plt.tight_layout()
plt.savefig(RESULT_DIR / "tabpfn_roc_pr.png", dpi=150)
plt.show()

## 6. Optional: stratified cross-validation (slow)

Each fold refits TabPFN on the training portion only. Use **`X_train_before_smote`** so SMOTE is not applied inconsistently across folds unless you intentionally resample inside a pipeline yourself.

In [ ]:
if RUN_TABPFN_CV:
    skf = StratifiedKFold(
        n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE
    )
    pr_aucs, roc_aucs, f1s = [], [], []
    for fold, (tr_idx, va_idx) in enumerate(
        skf.split(X_train_before_smote, y_train_before_smote), start=1
    ):
        X_tr, X_va = X_train_before_smote[tr_idx], X_train_before_smote[va_idx]
        y_tr, y_va = y_train_before_smote[tr_idx], y_train_before_smote[va_idx]
        m = TabPFNClassifier(
            n_estimators=N_ESTIMATORS,
            device=DEVICE,
            ignore_pretraining_limits=IGNORE_PRETRAINING_LIMITS,
            balance_probabilities=BALANCE_PROBABILITIES,
            random_state=RANDOM_STATE + fold,
        )
        m.fit(X_tr, y_tr)
        p = m.predict_proba(X_va)[:, 1]
        pred = m.predict(X_va)
        pr_aucs.append(average_precision_score(y_va, p))
        roc_aucs.append(roc_auc_score(y_va, p))
        f1s.append(f1_score(y_va, pred, zero_division=0))
        print(
            f"Fold {fold}/{CV_SPLITS}  PR-AUC={pr_aucs[-1]:.4f}  "
            f"ROC-AUC={roc_aucs[-1]:.4f}  F1={f1s[-1]:.4f}"
        )
    summary = pd.DataFrame(
        {
            "pr_auc_mean": np.mean(pr_aucs),
            "pr_auc_std": np.std(pr_aucs),
            "roc_auc_mean": np.mean(roc_aucs),
            "roc_auc_std": np.std(roc_aucs),
            "f1_mean": np.mean(f1s),
            "f1_std": np.std(f1s),
        },
        index=[0],
    )
    print(summary)
    summary.to_csv(RESULT_DIR / "tabpfn_cv_summary.csv", index=False)
else:
    print("Stratified CV skipped (set RUN_TABPFN_CV = True to enable).")